# EagleVision on SCARED / Endoscopy Stereo

This Kaggle notebook runs the controlled SCARED comparison:

1. Frozen Depth Anything V2-Small
2. Frozen DA2-Small + EagleVision tiny output residual log-depth adapter
3. DA2-Small + DARES-style internal LoRA baseline

The notebook expects the EagleVision repo to be available in Kaggle, plus a SCARED-style dataset mounted under `/kaggle/input`. Edit the paths in the first code cell if your dataset slug differs.

In [ ]:
from pathlib import Path
import os

# ---- Edit these for your Kaggle run ----
REPO_DIR = Path(os.environ.get("EAGLEVISION_REPO", "/kaggle/working/EagleVision"))
SCARED_ROOT = Path(os.environ.get("SCARED_ROOT", "/kaggle/input/SCARED"))

# Optional: set this if automatic left/right pairing cannot infer your SCARED layout.
# CSV columns: sample_id, sequence_id, frame_id, left_rgb, right_rgb, depth, disparity,
# mask, K_left, K_right, T_left_to_right, baseline, focal_length, split
MAPPING_CSV = os.environ.get("SCARED_MAPPING_CSV", "")

# Optional DA2 checkpoint. Leave as None to use whatever your local baseline config provides.
DA2_CHECKPOINT = os.environ.get("DA2_CHECKPOINT", "") or None

IMAGE_SIZE = [256, 320]
EPOCHS = 5
BATCH_SIZE = 4
NUM_WORKERS = 2

# Set QUICK_RUN=True for a smoke test before a real Kaggle run.
QUICK_RUN = False
MAX_STEPS_PER_EPOCH = 2 if QUICK_RUN else None

RUN_FROZEN_EVAL = True
RUN_ADAPTER_DRY_RUN = True
RUN_ADAPTER_TRAIN = True
RUN_LORA_TRAIN = True

if not REPO_DIR.exists() and Path.cwd().name == "EagleVision":
    REPO_DIR = Path.cwd()

print("Repo:", REPO_DIR)
print("SCARED root:", SCARED_ROOT)
print("Mapping CSV:", MAPPING_CSV or "<auto discovery>")
print("DA2 checkpoint:", DA2_CHECKPOINT or "<config/default>")
print("Quick run:", QUICK_RUN)

In [ ]:
import os, sys, subprocess, json, textwrap
from pathlib import Path

assert REPO_DIR.exists(), f"Repo not found: {REPO_DIR}"
os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

# Install EagleVision as an editable package. Kaggle usually already has torch installed.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
import yaml
from copy import deepcopy

CONFIG_DIR = Path("configs/endo")
MANIFEST_DIR = Path("manifests/scared")
RAW_RESULTS_DIR = Path("results/raw")
AGG_DIR = Path("results/aggregated/scared_head_to_head")

for path in [CONFIG_DIR, MANIFEST_DIR, RAW_RESULTS_DIR, AGG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def load_yaml(path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def save_yaml(path, payload):
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(payload, f, sort_keys=False)
    print("wrote", path)

def kaggleize(cfg, output_dir=None):
    cfg = deepcopy(cfg)
    cfg["device"] = "cuda" if torch.cuda.is_available() else "cpu"
    cfg.setdefault("data", {})
    cfg["data"]["root"] = str(SCARED_ROOT)
    cfg["data"]["image_size"] = IMAGE_SIZE
    cfg.setdefault("base_model", {})
    if DA2_CHECKPOINT:
        cfg["base_model"]["checkpoint_path"] = DA2_CHECKPOINT
    if "train" in cfg:
        cfg["train"]["epochs"] = EPOCHS
        cfg["train"]["batch_size"] = BATCH_SIZE
        cfg["train"]["num_workers"] = NUM_WORKERS
        cfg["train"]["max_steps_per_epoch"] = MAX_STEPS_PER_EPOCH
    cfg.setdefault("eval", {})
    cfg["eval"]["batch_size"] = 1
    cfg["eval"]["num_workers"] = NUM_WORKERS
    if output_dir is not None:
        cfg["output_dir"] = output_dir
    return cfg

frozen_cfg = kaggleize(load_yaml(CONFIG_DIR / "frozen_da2s.yaml"))
adapter_cfg = kaggleize(load_yaml(CONFIG_DIR / "eaglevision_da2s_cycle.yaml"), "outputs/endo/eaglevision_da2s_cycle")
lora_cfg = kaggleize(load_yaml(CONFIG_DIR / "lora_da2s_cycle_r4.yaml"), "outputs/endo/lora_da2s_cycle_r4")

save_yaml(CONFIG_DIR / "kaggle_frozen_da2s.yaml", frozen_cfg)
save_yaml(CONFIG_DIR / "kaggle_eaglevision_da2s_cycle.yaml", adapter_cfg)
save_yaml(CONFIG_DIR / "kaggle_lora_da2s_cycle_r4.yaml", lora_cfg)

## Build And Validate Manifests

Automatic discovery works only for common left/right folder naming. If this cell fails, create a mapping CSV and set `MAPPING_CSV` in the first cell.

In [ ]:
cmd = [
    sys.executable, "-m", "eaglevision.cli.build_scared_manifest",
    "--root", str(SCARED_ROOT),
    "--out-dir", str(MANIFEST_DIR),
    "--seed", "42",
]
if MAPPING_CSV:
    cmd.extend(["--mapping-csv", MAPPING_CSV])
print(" ".join(cmd))
subprocess.run(cmd, check=True)

for split in ["train", "val", "test"]:
    manifest = MANIFEST_DIR / f"{split}.jsonl"
    out = MANIFEST_DIR / f"{split}_validation.json"
    cmd = [
        sys.executable, "-m", "eaglevision.cli.validate_endoscopy_manifest",
        "--manifest", str(manifest),
        "--root", str(SCARED_ROOT),
        "--require-geometry",
        "--out", str(out),
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    print(split, json.loads(out.read_text()) | {"errors": "<omitted>"})

## Inspect DA2 Modules For LoRA Targets

In [ ]:
subprocess.run([
    sys.executable, "-m", "eaglevision.cli.inspect_da2_modules",
    "--config", str(CONFIG_DIR / "kaggle_frozen_da2s.yaml"),
    "--contains", "q", "proj", "attn", "linear", "mlp",
    "--out", "outputs/endo/da2_module_inspection.json",
], check=True)

## Frozen DA2-Small Evaluation

In [ ]:
if RUN_FROZEN_EVAL:
    subprocess.run([
        sys.executable, "-m", "eaglevision.cli.eval_endoscopy",
        "--config", str(CONFIG_DIR / "kaggle_frozen_da2s.yaml"),
        "--manifest", str(MANIFEST_DIR / "test.jsonl"),
        "--out", str(RAW_RESULTS_DIR / "frozen_da2s_scared.csv"),
    ], check=True)

## EagleVision Residual Adapter

In [ ]:
if RUN_ADAPTER_DRY_RUN:
    subprocess.run([
        sys.executable, "-m", "eaglevision.cli.train_endoscopy_adapter",
        "--config", str(CONFIG_DIR / "kaggle_eaglevision_da2s_cycle.yaml"),
        "--train-manifest", str(MANIFEST_DIR / "train.jsonl"),
        "--val-manifest", str(MANIFEST_DIR / "val.jsonl"),
        "--dry-run",
    ], check=True)

if RUN_ADAPTER_TRAIN:
    subprocess.run([
        sys.executable, "-m", "eaglevision.cli.train_endoscopy_adapter",
        "--config", str(CONFIG_DIR / "kaggle_eaglevision_da2s_cycle.yaml"),
        "--train-manifest", str(MANIFEST_DIR / "train.jsonl"),
        "--val-manifest", str(MANIFEST_DIR / "val.jsonl"),
    ], check=True)

adapter_ckpt = Path("outputs/endo/eaglevision_da2s_cycle/checkpoints/best.pt")
if adapter_ckpt.exists():
    subprocess.run([
        sys.executable, "-m", "eaglevision.cli.eval_endoscopy",
        "--config", str(CONFIG_DIR / "kaggle_eaglevision_da2s_cycle.yaml"),
        "--manifest", str(MANIFEST_DIR / "test.jsonl"),
        "--checkpoint", str(adapter_ckpt),
        "--out", str(RAW_RESULTS_DIR / "eaglevision_da2s_cycle_scared.csv"),
    ], check=True)
else:
    print("Adapter checkpoint not found; skipping adapter eval:", adapter_ckpt)

## DARES-Style LoRA Baseline

In [ ]:
if RUN_LORA_TRAIN:
    subprocess.run([
        sys.executable, "-m", "eaglevision.cli.train_endoscopy_lora",
        "--config", str(CONFIG_DIR / "kaggle_lora_da2s_cycle_r4.yaml"),
        "--train-manifest", str(MANIFEST_DIR / "train.jsonl"),
        "--val-manifest", str(MANIFEST_DIR / "val.jsonl"),
    ], check=True)

lora_ckpt = Path("outputs/endo/lora_da2s_cycle_r4/checkpoints/best.pt")
if lora_ckpt.exists():
    subprocess.run([
        sys.executable, "-m", "eaglevision.cli.eval_endoscopy",
        "--config", str(CONFIG_DIR / "kaggle_lora_da2s_cycle_r4.yaml"),
        "--manifest", str(MANIFEST_DIR / "test.jsonl"),
        "--checkpoint", str(lora_ckpt),
        "--out", str(RAW_RESULTS_DIR / "lora_da2s_cycle_r4_scared.csv"),
    ], check=True)
else:
    print("LoRA checkpoint not found; skipping LoRA eval:", lora_ckpt)

## Aggregate The Head-To-Head

In [ ]:
inputs = [
    RAW_RESULTS_DIR / "frozen_da2s_scared.csv",
    RAW_RESULTS_DIR / "eaglevision_da2s_cycle_scared.csv",
    RAW_RESULTS_DIR / "lora_da2s_cycle_r4_scared.csv",
]
inputs = [path for path in inputs if path.exists()]
param_summaries = [
    Path("outputs/endo/eaglevision_da2s_cycle/param_summary.json"),
    Path("outputs/endo/lora_da2s_cycle_r4/param_summary.json"),
]
param_summaries = [path for path in param_summaries if path.exists()]

cmd = [
    sys.executable, "scripts/aggregate_endoscopy_results.py",
    "--inputs", *map(str, inputs),
    "--baseline", "frozen_da2s",
    "--out-dir", str(AGG_DIR),
]
if param_summaries:
    cmd.extend(["--param-summaries", *map(str, param_summaries)])
print(" ".join(cmd))
subprocess.run(cmd, check=True)

print((AGG_DIR / "summary.md").read_text())
print((AGG_DIR / "method_ranking.md").read_text()[:4000])

## Preview Results And Panels

In [ ]:
import pandas as pd
from IPython.display import display, Image

summary_csv = AGG_DIR / "summary.csv"
if summary_csv.exists():
    display(pd.read_csv(summary_csv))

for panel in [
    Path("outputs/endo/eaglevision_da2s_cycle/panels/dry_run.png"),
    Path("outputs/endo/eaglevision_da2s_cycle/panels/epoch_001.png"),
    Path("outputs/endo/lora_da2s_cycle_r4/panels/epoch_001.png"),
]:
    if panel.exists():
        print(panel)
        display(Image(filename=str(panel)))